# ANIMA-MECHANICUS — Fregate U-ALPHA : L'Auspex Cogitateur
> *L'Ame du Mouvement, Forgee par la Machine*

**Pipeline** : `.mp4 (humain reel)` → `[Gemini Chat]` → `[WHAM SMPL]` → `[Retargeting R15]` → `.npz`

**Instructions (mode chat — sans cle API)** :
1. Cellule 1 — Installer les dependances (WHAM, Torch, Detectron2, google-genai)
2. Cellule 1b — Copier SMPL_NEUTRAL.pkl depuis Google Drive *(si manquant au Pre-Flight)*
3. Cellule 2 — Configurer les parametres pipeline *(GEMINI_API_KEY optionnel en mode chat)*
4. Cellule 3 — Pre-flight check (verifier que tout est pret)
5. Cellule 4 — Uploader la video `.mp4` source
6. Cellule 4b — Coller le JSON obtenu depuis Gemini Chat ([gemini.google.com](https://gemini.google.com))
7. Cellule 6 — Verifier le routing par segment
8. Cellule 7 — Lancer WHAM → SMPL→R15 → exporter `.npz`
9. Cellule 8 — Telecharger les `.npz` (input de la Fregate U-GAMMA)

> **Prompt Gemini Chat** : voir `GEMINI_CHAT_METAPROMPT.md` dans le depot

**Runtime requis** : GPU T4 (Colab gratuit)

---

In [ ]:
# ══════ CELLULE 1 — INSTALLATION ══════
# WHAM + ViTPose + Torch CUDA + Detectron2 + Google Generative AI
# Duree estimee : 10-15 min sur Colab T4 (1 seule fois par session)

import os, subprocess, sys

def xpip(spec, extra_flags=""):
    """pip install robuste : detecte erreur setup.py et affiche le vrai message."""
    cmd = f"pip install -q {extra_flags} {spec}"
    ret = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if ret.returncode != 0:
        stderr = ret.stderr
        if "setup.py egg_info" in stderr or "metadata-generation-failed" in stderr:
            print(f"  WARN setup.py legacy — retry --no-build-isolation")
            ret2 = subprocess.run(
                f"pip install -q --no-build-isolation {extra_flags} {spec}",
                shell=True, capture_output=True, text=True
            )
            if ret2.returncode != 0:
                print(f"  FAIL: {ret2.stderr[-300:].strip()}")
                return False
        else:
            print(f"  FAIL: {stderr[-300:].strip()}")
            return False
    return True

print("[1/9] ffmpeg + libs systeme...")
os.system("apt-get install -y ffmpeg libgl1 > /dev/null 2>&1")
print("  OK")

print("[2/9] PyTorch CUDA 11.8...")
xpip("torch==2.1.0 torchvision==0.16.0",
     extra_flags="--index-url https://download.pytorch.org/whl/cu118")
print("  OK")

print("[3/9] mmcv-full 1.3.9...")
xpip("openmim")
subprocess.run("python -m mim install -q \'mmcv-full==1.3.9\'", shell=True)
print("  OK")

print("[4/9] Detectron2 (wheel pre-compile — evite setup.py)...")
# Utiliser le wheel pre-compile Meta au lieu du source (pas de compilation requise)
ret = subprocess.run(
    "pip install -q detectron2 "
    "-f https://dl.fbaipublicfiles.com/detectron2/wheels/cu118/torch2.1/index.html",
    shell=True
)
if ret.returncode != 0:
    print("  WARN wheel indisponible — tentative source...")
    xpip("git+https://github.com/facebookresearch/detectron2.git")
print("  OK")

print("[5/9] google-genai...")
xpip("google-genai")
print("  OK")

print("[6/9] WHAM + ViTPose...")
WHAM_DIR = os.path.expanduser("~/WHAM")
if not os.path.exists(WHAM_DIR):
    os.system("git clone -q --recursive https://github.com/yohanshin/WHAM.git ~/WHAM")
else:
    os.system("cd ~/WHAM && git submodule update --init --recursive -q")

xpip("setuptools wheel")

# numpy.distutils supprime dans numpy>=1.24 — downgrader avant les packages legacy
print("  numpy legacy compat...")
xpip("\'numpy==1.23.5\'")

# chumpy utilise numpy.distutils (setup.py legacy)
xpip("git+https://github.com/mattloper/chumpy")

# requirements.txt (contient numpy==1.22.3, xtcocotools, setuptools==59.5.0)
subprocess.run(
    "pip install -q --no-build-isolation -r ~/WHAM/requirements.txt",
    shell=True
)

# ViTPose editable
subprocess.run(
    "pip install -q --no-build-isolation -e /root/WHAM/third-party/ViTPose",
    shell=True
)

# Deps supplementaires
xpip("loguru progress yacs einops munkres smplx \'timm==0.4.9\' gdown "
     "ultralytics joblib \'imageio[ffmpeg]\' tensorboard")
print("  WHAM + ViTPose OK")

print("[7/9] Checkpoints WHAM...")
CKPT_DIR = os.path.expanduser("~/WHAM/checkpoints")
os.makedirs(CKPT_DIR, exist_ok=True)
CHECKPOINTS = {
    "wham_vit_w_3dpw.pth.tar": "1i7kt9RlCCCNEW2aYaDWVr-G778JkLNcB",
    "yolov8x.pt":               "1zJ0KP23tXD42D47cw1Gs7zE2BA_V_ERo",
    "vitpose-h-multi-coco.pth": "1xyF7F3I7lWtdq82xmEPVQ5zl4HaasBso",
}
for fname, gdid in CHECKPOINTS.items():
    dst = os.path.join(CKPT_DIR, fname)
    if not os.path.exists(dst):
        print(f"  Telechargement {fname}...")
        os.system(f"gdown \'https://drive.google.com/uc?id={gdid}&export=download&confirm=t\' -O \'{dst}\' -q")
        sz = os.path.getsize(dst)/1024/1024 if os.path.exists(dst) else 0
        print(f"  OK {fname} ({sz:.0f} MB)" if sz > 1 else f"  WARN {fname} ({sz:.1f} MB)")
    else:
        sz = os.path.getsize(dst)/1024/1024
        print(f"  OK {fname} deja present ({sz:.0f} MB)")

print("[8/9] ANIMA-MECHANICUS...")
REPO_DIR = "/content/ANIMA-MECHANICUS"
if os.path.exists(REPO_DIR):
    os.system(f"cd {REPO_DIR} && git pull -q")
else:
    os.system(f"git clone -q https://github.com/kioka8877-ux/ANIMA-MECHANICUS.git {REPO_DIR}")
xpip("numpy scipy opencv-python-headless \'scenedetect[opencv]\'")
print("  OK")

print("\n[U-ALPHA] Installation terminee.")
print("IMPORTANT — Modeles SMPL requis si pas encore fait :")
print("  1. https://smpl.is.tue.mpg.de/ -> SMPL_python_v.1.1.0.zip")
print("  2. Extraire SMPL_NEUTRAL.pkl -> /content/body_models/smpl/SMPL_NEUTRAL.pkl")


In [ ]:
# ══════ CELLULE 1b — SMPL DEPUIS GOOGLE DRIVE ══════
# Monte ton Drive, extrait SMPL_NEUTRAL.pkl du zip et le copie au bon endroit
# Executer seulement si le Pre-Flight signale que SMPL_NEUTRAL.pkl est manquant

import os, shutil, zipfile, time
from google.colab import drive

print("[SMPL] Montage de Google Drive...")
drive.mount("/content/drive", force_remount=False)

DST = "/content/body_models/smpl/SMPL_NEUTRAL.pkl"
os.makedirs(os.path.dirname(DST), exist_ok=True)

DRIVE_ROOT = "/content/drive/My Drive"

# ── Helper : recherche limitee pour eviter les 429 Drive API ────────────────
def _find_in_drive(root, filename_contains, extension, max_depth=3):
    """Cherche un fichier dans Drive sans traverser toute l'arborescence.

    Strategie :
      1. Racine du Drive (depth 0)
      2. Sous-dossiers directs de la racine (depth 1)
      3. Profondeur max_depth en dernier recours
    Ajoute des pauses entre les lectures de dossiers pour eviter le 429 Drive API.
    """
    found = []

    def _scan(path, depth):
        if depth > max_depth:
            return
        try:
            entries = os.listdir(path)
        except PermissionError:
            return
        time.sleep(0.05)  # pause legere : evite le 429 sur les grands Drives
        for name in entries:
            full = os.path.join(path, name)
            if os.path.isfile(full):
                n_lower = name.lower()
                if filename_contains.lower() in n_lower and n_lower.endswith(extension):
                    found.append(full)
            elif os.path.isdir(full) and depth < max_depth:
                _scan(full, depth + 1)

    _scan(root, 0)
    return found

# --- Cas 1 : SMPL_NEUTRAL.pkl deja present dans le Drive ---
print("[SMPL] Recherche de SMPL_NEUTRAL.pkl dans Drive (profondeur 3)...")
found_pkl = _find_in_drive(DRIVE_ROOT, "SMPL_NEUTRAL", ".pkl")

if found_pkl:
    shutil.copy2(found_pkl[0], DST)
    size_mb = os.path.getsize(DST) / 1024 / 1024
    print(f"[SMPL] OK — copie depuis {found_pkl[0]} ({size_mb:.1f} MB)")

else:
    # --- Cas 2 : zip SMPL present dans le Drive → extraire ---
    print("[SMPL] .pkl non trouve — recherche du zip SMPL...")
    found_zip = _find_in_drive(DRIVE_ROOT, "smpl", ".zip")

    if not found_zip:
        print("[SMPL] Aucun zip SMPL trouve dans Drive.")
        print("  Telecharger depuis : https://smpl.is.tue.mpg.de/ (inscription gratuite)")
        print("  Deposer SMPL_python_v.1.1.0.zip dans Mon Drive puis relancer cette cellule.")
        raise FileNotFoundError("SMPL zip introuvable dans Google Drive")

    zip_path = found_zip[0]
    print(f"[SMPL] Zip trouve : {zip_path}")
    print("[SMPL] Extraction de SMPL_NEUTRAL.pkl...")

    with zipfile.ZipFile(zip_path, 'r') as z:
        pkl_files = [f for f in z.namelist() if f.endswith(".pkl") and "NEUTRAL" in f.upper()]
        if not pkl_files:
            pkl_files = [f for f in z.namelist() if f.endswith(".pkl")]
        if not pkl_files:
            print("[SMPL] Contenu du zip :")
            for name in z.namelist():
                print(f"  {name}")
            raise FileNotFoundError("Aucun .pkl trouve dans le zip")

        import tempfile
        with tempfile.TemporaryDirectory() as tmp:
            z.extract(pkl_files[0], tmp)
            shutil.copy2(os.path.join(tmp, pkl_files[0]), DST)

    size_mb = os.path.getsize(DST) / 1024 / 1024
    print(f"[SMPL] OK — extrait depuis zip ({size_mb:.1f} MB)")

print(f"[SMPL] Fichier pret : {DST}")
print("[SMPL] Relancer la Cellule 3 (Pre-Flight Check)")


In [ ]:
# ══════ CELLULE 2 — CONFIGURATION ══════

import os

# @title Parametres U-ALPHA

GEMINI_API_KEY      = ""       # @param {type:"string"} — obtenir sur https://aistudio.google.com
FPS_CIBLE           = 30       # @param [30, 60, 120] {type:"raw"}
LISSAGE             = "moyen" # @param ["faible", "moyen", "brutal"] {type:"string"}
ROOT_MOTION         = True     # @param {type:"boolean"}
QUALITY_THRESHOLD   = 0.6      # @param {type:"number"}
SMPL_MODEL_PATH     = "/content/body_models/smpl/SMPL_NEUTRAL.pkl"  # @param {type:"string"}

# Cache Gemini — evite de re-consommer du quota sur la meme video
# Laisser vide "" pour desactiver, ou garder le chemin par defaut
GEMINI_CACHE_DIR    = "/content/gemini_cache"  # @param {type:"string"}

WHAM_DIR   = os.path.expanduser("~/WHAM")
REPO_DIR   = "/content/ANIMA-MECHANICUS"
OUTPUT_DIR = "/content/alpha_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
if GEMINI_CACHE_DIR:
    os.makedirs(GEMINI_CACHE_DIR, exist_ok=True)

print(f"[CONFIG] FPS={FPS_CIBLE} | Lissage={LISSAGE} | RootMotion={ROOT_MOTION} | Seuil={QUALITY_THRESHOLD}")
print(f"[CONFIG] Cache Gemini : {GEMINI_CACHE_DIR if GEMINI_CACHE_DIR else 'desactive'}")
print("[U-ALPHA] Configuration sauvegardee — passer a la Cellule 3 (pre-flight check)")


In [ ]:
# ══════ CELLULE 3 — PRE-FLIGHT CHECK ══════
# Verifie que toutes les dependances sont operationnelles avant de lancer WHAM

import os, sys, subprocess

errors  = []
warnings = []

print("[PRE-FLIGHT] Verification de l'environnement...\n")

# 1. GPU
gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                             "--format=csv,noheader"], capture_output=True, text=True)
if gpu_check.returncode == 0:
    gpu_info = gpu_check.stdout.strip()
    print(f"  OK  GPU : {gpu_info}")
else:
    errors.append("GPU non detecte — activer le runtime GPU T4 dans Colab (Execution → Modifier le type d'execution)")

# 2. PyTorch + CUDA
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"  OK  PyTorch {torch.__version__} | CUDA : {'oui' if cuda_ok else 'NON'}")
    if not cuda_ok:
        warnings.append("PyTorch sans CUDA — WHAM sera tres lent (CPU only)")
except ImportError:
    errors.append("PyTorch non installe — relancer la Cellule 1")

# 3. google-genai
try:
    from google import genai as _genai_check
    print(f"  OK  google-genai {_genai_check.__version__}")
except ImportError:
    errors.append("google-genai non installe — relancer la Cellule 1")

# 4. Cle Gemini — OPTIONNELLE en mode chat (Cellule 4b)
if not GEMINI_API_KEY:
    print("  INFO Gemini API key absente — mode chat actif (Cellule 4b)")
    print("       Cle non requise si tu utilises Gemini Chat + Cellule 4b.")
else:
    try:
        from google import genai as _genai_val
        _client = _genai_val.Client(api_key=GEMINI_API_KEY)
        list(_client.models.list())
        print("  OK  Gemini API key valide")
    except Exception as e:
        warnings.append(f"Gemini API key invalide ou reseau KO : {e} — utilise la Cellule 4b a la place")

# 5. WHAM
if not os.path.exists(WHAM_DIR):
    errors.append(f"WHAM introuvable : {WHAM_DIR} — relancer Cellule 1")
else:
    demo = os.path.join(WHAM_DIR, "demo.py")
    if os.path.exists(demo):
        print(f"  OK  WHAM : {WHAM_DIR}")
    else:
        errors.append(f"WHAM incomplet (demo.py absent) — re-cloner : git clone https://github.com/yohanshin/WHAM.git ~/WHAM")

# 6. Modele SMPL
if not os.path.exists(SMPL_MODEL_PATH):
    errors.append(
        f"Modele SMPL introuvable : {SMPL_MODEL_PATH}\n"
        "  → S'inscrire sur https://smpl.is.tue.mpg.de/ et uploader SMPL_NEUTRAL.pkl"
    )
else:
    size_mb = os.path.getsize(SMPL_MODEL_PATH) / 1024 / 1024
    if size_mb < 1:
        errors.append(f"Modele SMPL trop petit ({size_mb:.1f} MB) — fichier corrompu ou incomplet")
    else:
        print(f"  OK  Modele SMPL : {SMPL_MODEL_PATH} ({size_mb:.1f} MB)")

# 7. scipy / numpy / cv2
for pkg in ["numpy", "scipy", "cv2"]:
    try:
        mod = __import__(pkg)
        ver = getattr(mod, "__version__", "?")
        print(f"  OK  {pkg} {ver}")
    except ImportError:
        errors.append(f"{pkg} non installe — relancer Cellule 1")

# 8. ffmpeg
ffmpeg_check = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if ffmpeg_check.returncode == 0:
    version_line = ffmpeg_check.stdout.splitlines()[0]
    print(f"  OK  {version_line[:50]}")
else:
    errors.append("ffmpeg non installe — relancer Cellule 1")

# 9. Codebase ANIMA-MECHANICUS
extract_script = f"{REPO_DIR}/U-ALPHA/codebase/motus_extract.py"
if os.path.exists(extract_script):
    print(f"  OK  motus_extract.py present")
else:
    errors.append(f"motus_extract.py introuvable — relancer Cellule 1")

# ── Bilan ─────────────────────────────────────────────────────────────────
print()
if warnings:
    print("AVERTISSEMENTS :")
    for w in warnings:
        print(f"  [WARN] {w}")
    print()

if errors:
    print(f"PRE-FLIGHT ECHEC — {len(errors)} erreur(s) a corriger :")
    for i, e in enumerate(errors, 1):
        print(f"  [{i}] {e}")
    raise RuntimeError("Corriger les erreurs ci-dessus avant de continuer")
else:
    print("PRE-FLIGHT OK — Tous les systemes sont operationnels.")
    if not GEMINI_API_KEY:
        print("  Mode chat : utilise Cellule 4b pour injecter le JSON Gemini.")
    else:
        print("Passer a la Cellule 4 pour uploader la video.")

In [ ]:
# ══════ CELLULE 4 — UPLOAD VIDEO ══════
# Uploader la video .mp4 source (humain reel uniquement, max 60 secondes)

from google.colab import files as colab_files
import os, cv2

VIDEO_PATH = None

print("Option A — Upload direct depuis votre ordinateur :")
try:
    uploaded = colab_files.upload()
    for fname, data in uploaded.items():
        if fname.lower().endswith(".mp4"):
            VIDEO_PATH = f"/content/{fname}"
            with open(VIDEO_PATH, "wb") as f:
                f.write(data)
            break
except Exception:
    pass

if not VIDEO_PATH:
    VIDEO_PATH_MANUEL = ""  # @param {type:"string"}
    if VIDEO_PATH_MANUEL and os.path.exists(VIDEO_PATH_MANUEL):
        VIDEO_PATH = VIDEO_PATH_MANUEL

if not VIDEO_PATH or not os.path.exists(VIDEO_PATH):
    raise ValueError("[ERREUR] Aucune video .mp4 chargee")

cap = cv2.VideoCapture(VIDEO_PATH)
src_fps    = cap.get(cv2.CAP_PROP_FPS)
tot_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration   = tot_frames / src_fps if src_fps > 0 else 0
cap.release()
size_mb = os.path.getsize(VIDEO_PATH) / 1024 / 1024

print(f"  OK  {os.path.basename(VIDEO_PATH)}")
print(f"      {width}x{height} | {src_fps:.1f} FPS | {duration:.1f}s | {size_mb:.1f} MB")

if duration > 60:
    print(f"  [WARN] Duree {duration:.1f}s > 60s — pipeline prevu pour max 60 secondes")
if width < 480 or height < 480:
    print(f"  [WARN] Resolution faible ({width}x{height}) — qualite WHAM potentiellement reduite")

In [ ]:
# ══════ CELLULE 4b — INJECTION JSON GEMINI (MODE CHAT) ══════
# Utilise cette cellule si tu as obtenu le JSON depuis gemini.google.com (chat)
# Apres validation, ALL_SEGMENTS est pret — passe directement a la Cellule 6.
# Cellule 5 (API) sera automatiquement ignoree.

import sys, os, hashlib
import json as _json
import ipywidgets as widgets
from IPython.display import display

sys.path.insert(0, f"{REPO_DIR}/U-ALPHA/codebase")

assert VIDEO_PATH and os.path.exists(VIDEO_PATH), "[ERREUR] Charger une video en Cellule 4 d'abord"

_ta = widgets.Textarea(
    placeholder='Colle ici le JSON complet fourni par Gemini Chat...',
    layout=widgets.Layout(width='100%', height='280px')
)
_btn = widgets.Button(description='Valider et injecter', button_style='success',
                      layout=widgets.Layout(width='200px'))
_out = widgets.Output()

def _on_inject(b):
    global ALL_SEGMENTS, GEMINI_JSON_INJECTED, GEMINI_DATA
    with _out:
        _out.clear_output()
        raw = _ta.value.strip()
        if not raw:
            print("[SKIP] Zone vide — rien a injecter")
            return
        try:
            data = _json.loads(raw)
        except _json.JSONDecodeError as e:
            print(f"[ERREUR] JSON invalide : {e}")
            return

        # Sauvegarder en cache (pour reutilisation future)
        cache_dir = GEMINI_CACHE_DIR or "/content/gemini_cache"
        os.makedirs(cache_dir, exist_ok=True)
        with open(VIDEO_PATH, "rb") as f:
            h = hashlib.md5(f.read(1024 * 1024)).hexdigest()[:10]
        stem = os.path.splitext(os.path.basename(VIDEO_PATH))[0]
        cache_path = os.path.join(cache_dir, f"gemini_cache_{stem}_{h}.json")
        with open(cache_path, "w", encoding="utf-8") as f:
            _json.dump(data, f, ensure_ascii=False, indent=2)

        # Filtrer les segments
        from motus_extract import filter_segments
        segs = filter_segments(data, QUALITY_THRESHOLD)

        # Injecter dans les globales
        GEMINI_DATA          = data
        ALL_SEGMENTS         = segs
        GEMINI_JSON_INJECTED = True

        # Rapport
        print(f"  JSON valide : {len(data['persons'])} personne(s) | "
              f"{data['video_duration_seconds']}s | qualite={data['qualite_globale']}")
        print()
        for p in data['persons']:
            print(f"  Personne {p['person_id']} :")
            for seg in p.get('segments_valides', []):
                modele = seg.get('modele_recommande', '?')
                extraction = seg.get('extraction_possible', '?')
                dur = seg['end_s'] - seg['start_s']
                print(f"    [{seg['start_s']}s -> {seg['end_s']}s ({dur:.1f}s)] "
                      f"{seg['corps_visible']} -> {modele} ({extraction})")
            for seg in p.get('segments_exclus', []):
                print(f"    [{seg['start_s']}s -> {seg['end_s']}s] EXCLU : {seg['raison']}")
        print()
        print(f"  ALL_SEGMENTS : {len(segs)} segment(s) retenus (seuil >= {QUALITY_THRESHOLD})")
        print(f"  Cache sauvegarde : {cache_path}")
        print()
        print("  OK — Passe directement a la Cellule 6 — Cellule 5 sera ignoree automatiquement")

_btn.on_click(_on_inject)
display(widgets.VBox([_ta, _btn, _out]))


In [ ]:
# ══════ CELLULE 5 — ANALYSE GEMINI API (AUTO) ══════
# A utiliser UNIQUEMENT si tu n'as PAS utilise la Cellule 4b.
# Si ALL_SEGMENTS est deja defini (via Cellule 4b), cette cellule se skipe.

import sys, os
sys.path.insert(0, f"{REPO_DIR}/U-ALPHA/codebase")
from motus_extract import analyze_video_gemini, print_gemini_report, filter_segments

# ── Garde : skip si JSON deja injecte via 4b ──────────────────────────
if globals().get('GEMINI_JSON_INJECTED'):
    print("[U-ALPHA][Cellule 5] JSON deja injecte via Cellule 4b — cellule ignoree")
    print(f"  ALL_SEGMENTS : {len(ALL_SEGMENTS)} segment(s) deja disponibles")
    print("  Passe a la Cellule 6.")
else:
    assert GEMINI_API_KEY, "[ERREUR] Configurer GEMINI_API_KEY en Cellule 2 (ou utiliser Cellule 4b)"
    assert VIDEO_PATH and os.path.exists(VIDEO_PATH), "[ERREUR] Charger une video en Cellule 4"

    print("[U-ALPHA] Analyse Gemini API (cascade auto)...")
    if GEMINI_CACHE_DIR:
        print(f"[U-ALPHA] Cache actif : {GEMINI_CACHE_DIR}")

    GEMINI_DATA = analyze_video_gemini(
        VIDEO_PATH,
        GEMINI_API_KEY,
        cache_dir=GEMINI_CACHE_DIR,
    )
    print_gemini_report(GEMINI_DATA)

    ALL_SEGMENTS = filter_segments(GEMINI_DATA, QUALITY_THRESHOLD)

    print(f"{'='*60}")
    print(f"[FILTRE] {len(ALL_SEGMENTS)} segment(s) retenus (seuil >= {QUALITY_THRESHOLD}) :")
    for s in ALL_SEGMENTS:
        dur = s['end_s'] - s['start_s']
        modele = s.get('modele_recommande', 'WHAM')
        print(f"  P{s['person_id']} | {s['start_s']:.1f}s-{s['end_s']:.1f}s ({dur:.1f}s) | "
              f"{s['corps_visible']} | qualite={s['qualite_estimee']:.2f} | {modele}")
    print(f"{'='*60}")

    cam = GEMINI_DATA.get("camera", {})
    if cam.get("mouvement") == "agitee":
        print("[WARN] Camera agitee — root motion peut etre peu fiable")
    if cam.get("zoom_detecte"):
        print("[WARN] Zoom detecte — root motion peut etre fausse")
    if not ALL_SEGMENTS:
        print("[ERREUR] Aucun segment valide — verifier que la video contient un humain clairement visible")


In [ ]:
# ══════ CELLULE 6 — VALIDATION DES SEGMENTS ══════

# === Ajustements manuels optionnels ===
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s['person_id'] == 1]
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s.get('modele_recommande') == 'WHAM']
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s['start_s'] >= 5.0]
# SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s['corps_visible'] == 'complet']

SEGMENTS_TO_PROCESS = ALL_SEGMENTS

# ── Routing par modèle ───────────────────────────────────────────────
_wham_segs   = [s for s in SEGMENTS_TO_PROCESS if not s.get('_skip_wham')]
_true_wham   = [s for s in _wham_segs if s.get('modele_recommande', 'WHAM') == 'WHAM']
_frank_segs  = [s for s in _wham_segs if s.get('modele_recommande') == 'FrankMocap_upper']
_deca_segs   = [s for s in SEGMENTS_TO_PROCESS if s.get('_skip_wham')]
_true_dur    = sum(s['end_s'] - s['start_s'] for s in _true_wham)
_frank_dur   = sum(s['end_s'] - s['start_s'] for s in _frank_segs)

print(f"[U-ALPHA] Routing des {len(SEGMENTS_TO_PROCESS)} segment(s) valides :")
print()

MODELE_LABEL = {
    'WHAM':             '\u2705 WHAM          ',
    'FrankMocap_upper': '\u26a0\ufe0f  FrankMocap    ',
    'DECA':             '\u26a0\ufe0f  DECA          ',
    'skip':             '\u23ed\ufe0f  SKIP          ',
}

for s in SEGMENTS_TO_PROCESS:
    dur = s['end_s'] - s['start_s']
    modele = s.get('modele_recommande', 'WHAM')
    extraction = s.get('extraction_possible', '?')
    label = MODELE_LABEL.get(modele, modele)
    skip_tag = ' [sera ignore par WHAM]' if s.get('_skip_wham') else ''
    print(f"  {label} | P{s['person_id']} {s['start_s']:.1f}s\u2192{s['end_s']:.1f}s "
          f"({dur:.1f}s) | {s['corps_visible']} | {extraction}{skip_tag}")

print()
print(f"  Segments WHAM (corps_complet)       : {len(_true_wham)} ({_true_dur:.1f}s)")
if _frank_segs:
    print(f"  Segments FrankMocap \u2192 fallback WHAM : {len(_frank_segs)} ({_frank_dur:.1f}s)")
    print(f"    [WARN] WHAM va halluciner les membres inferieurs sur ces segments")
    print(f"           Verifier visuellement les .npz produits avant envoi a U-GAMMA")
if _deca_segs:
    print(f"  Segments DECA (ignores)             : {len(_deca_segs)}")
print()

if not _wham_segs:
    print("[WARN] Aucun segment WHAM disponible.")
    if _deca_segs:
        print("       Tous les segments sont DECA/FrankMocap — non encore implementes.")
        print("       Pour forcer WHAM sur les segments tronc : decommente la ligne ci-dessus")
        print("       SEGMENTS_TO_PROCESS = [s for s in ALL_SEGMENTS if s.get('modele_recommande') == 'WHAM']")
    raise ValueError("Aucun segment WHAM — ajuste la selection ci-dessus")
else:
    total_wham = len(_wham_segs)
    print(f"  OK — {total_wham} segment(s) envoyes a WHAM. Passer a la Cellule 7.")


In [ ]:
# ══════ CELLULE 6b — VERIFICATION DES DEPENDANCES ══════
# A lancer si Cellule 7 plante sur un ModuleNotFoundError.
# Verifie que tous les packages requis par WHAM sont bien installes
# et installe les manquants sans re-executer toute la Cellule 1.

import sys, subprocess

# Tuple : (nom_import, commande_pip_ou_None)
# None = reinstallation manuelle requise (torch, mmpose, cv2...)
# Prefixer par "--no-build-isolation " pour les packages setup.py legacy
REQUIRED = [
    ("loguru",       "loguru"),
    ("progress",     "progress"),
    ("yacs",         "yacs"),
    ("einops",       "einops"),
    ("munkres",      "munkres"),
    ("smplx",        "smplx"),
    ("timm",         "timm"),
    ("gdown",        "gdown"),
    ("ultralytics",  "ultralytics"),
    ("joblib",       "joblib"),
    ("imageio",      "imageio[ffmpeg]"),
    ("tensorboard",  "tensorboard"),
    ("chumpy",       "--no-build-isolation git+https://github.com/mattloper/chumpy"),
    ("torch",        None),
    ("cv2",          None),
    ("scipy",        None),
    ("numpy",        None),
    ("mmpose",       None),
]

missing_to_install = []
already_ok         = []
skip_reinstall     = []

for import_name, pip_name in REQUIRED:
    try:
        __import__(import_name)
        already_ok.append(import_name)
    except ImportError:
        if pip_name is None:
            skip_reinstall.append(import_name)
        else:
            missing_to_install.append((import_name, pip_name))

print(f"[6b] OK ({len(already_ok)}) : {', '.join(already_ok)}")

if skip_reinstall:
    print(f"[6b] MANQUANTS (reinstallation manuelle) : {', '.join(skip_reinstall)}")
    if "mmpose" in skip_reinstall:
        print("     mmpose = fourni par ViTPose.  Lancer dans une cellule :")
        print("       !pip install -q --no-build-isolation -e /root/WHAM/third-party/ViTPose")
    if "cv2" in skip_reinstall:
        print("     cv2 manquant : !pip install -q opencv-python-headless")
    if "torch" in skip_reinstall:
        print("     torch manquant : relancer la Cellule 1 complete")

if missing_to_install:
    print(f"\n[6b] Installation des {len(missing_to_install)} package(s) manquant(s)...")
    for import_name, pip_spec in missing_to_install:
        # Separer les flags eventuels (ex: "--no-build-isolation git+https://...")
        parts = pip_spec.split()
        cmd = [sys.executable, "-m", "pip", "install", "-q"] + parts
        print(f"  {import_name} ...", end=" ", flush=True)
        ret = subprocess.run(cmd, capture_output=True)
        print("OK" if ret.returncode == 0 else f"FAIL\n    {ret.stderr.decode()[-200:].strip()}")
    print("\n[6b] Installation terminee — relancer Cellule 7.")
elif not skip_reinstall:
    print("\n[6b] Toutes les dependances presentes — Cellule 7 prete.")
else:
    print("\n[6b] Corriger les manquants ci-dessus avant de lancer Cellule 7.")


In [ ]:
# ══════ CELLULE 7 — EXTRACTION WHAM → SMPL→R15 → .npz ══════
# Duree estimee : 1-3 min par segment de 30s sur Colab T4

import sys, os, tempfile
import numpy as np
import cv2

sys.path.insert(0, f"{REPO_DIR}/U-ALPHA/codebase")
from motus_extract import (
    run_wham, smpl_to_r15, fill_gaps,
    smooth_array, smooth_extremities,
    temporal_resample, normalize_quats,
    export_npz, SMOOTH_PRESETS
)

assert SEGMENTS_TO_PROCESS, "[ERREUR] Valider les segments en Cellule 6"

# Filtrer uniquement les segments WHAM (exclure DECA et FrankMocap non implementes)
WHAM_SEGMENTS = [s for s in SEGMENTS_TO_PROCESS if not s.get('_skip_wham')]
PENDING_SEGMENTS = [s for s in SEGMENTS_TO_PROCESS if s.get('_skip_wham')]

if not WHAM_SEGMENTS:
    raise ValueError("[ERREUR] Aucun segment WHAM — relancer Cellule 6")

if PENDING_SEGMENTS:
    print(f"[INFO] {len(PENDING_SEGMENTS)} segment(s) DECA/FrankMocap ignores — non implementes")
    for s in PENDING_SEGMENTS:
        print(f"  P{s['person_id']} {s['start_s']:.1f}s→{s['end_s']:.1f}s | {s.get('modele_recommande','?')}")
    print()

# WARN si segments FrankMocap traites en fallback WHAM
_frank_fallback = [s for s in WHAM_SEGMENTS if s.get('modele_recommande') == 'FrankMocap_upper']
if _frank_fallback:
    print(f"[WARN] {len(_frank_fallback)} segment(s) FrankMocap traites par WHAM en fallback")
    print(f"       Corps = tronc — membres inferieurs hallucines dans le .npz produit")
    print(f"       Verifier visuellement avant envoi a U-GAMMA")
    print()

cap = cv2.VideoCapture(VIDEO_PATH)
src_fps    = cap.get(cv2.CAP_PROP_FPS)
tot_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration   = tot_frames / src_fps
cap.release()
print(f"[U-ALPHA] Source : {src_fps:.1f} FPS, {duration:.1f}s")
print(f"[U-ALPHA] {len(WHAM_SEGMENTS)} segment(s) WHAM a traiter")

# Etape 1 : WHAM
print("\n[U-ALPHA] 1/3 — Extraction WHAM (poses SMPL)...")
with tempfile.TemporaryDirectory() as tmp_dir:
    wham_tracks = run_wham(VIDEO_PATH, WHAM_SEGMENTS, WHAM_DIR, tmp_dir)

if not wham_tracks:
    raise RuntimeError("[ERREUR] WHAM n'a produit aucun resultat")

# Etapes 2-3 : Retargeting + Lissage + Export
print("\n[U-ALPHA] 2/3 — Retargeting SMPL→R15 + Lissage...")
win, poly = SMOOTH_PRESETS[LISSAGE]
NPZ_FILES = []
n_persons = len(wham_tracks)

for pid in sorted(wham_tracks.keys()):
    track = wham_tracks[pid]
    print(f"  Personne {pid} : {len(track['poses'])} frames brutes")

    fill_gaps(track["poses"], track["transl"])
    indices  = sorted(track["poses"].keys())
    poses_aa = np.array([track["poses"][i] for i in indices])
    transl   = np.array([track["transl"][i] for i in indices])

    rots, root_pos = smpl_to_r15(poses_aa, transl)
    if not ROOT_MOTION:
        root_pos = np.zeros_like(root_pos)

    root_pos = smooth_array(root_pos, win, poly)
    rots = smooth_array(rots.reshape(len(rots), -1), win, poly).reshape(-1, 15, 4)
    rots = normalize_quats(rots)
    rots = smooth_extremities(rots, max(win * 2 + 1, 15))

    root_pos = temporal_resample(root_pos, src_fps, FPS_CIBLE)
    rots = temporal_resample(rots.reshape(len(rots), -1), src_fps, FPS_CIBLE).reshape(-1, 15, 4)
    rots = normalize_quats(rots)

    # Validation avant sauvegarde
    assert rots.shape == (len(rots), 15, 4), f"Shape rotations incorrecte : {rots.shape}"
    assert not np.any(np.isnan(rots)), "NaN detecte dans les rotations"
    assert not np.any(np.isnan(root_pos)), "NaN detecte dans root_position"

    out_path = f"{OUTPUT_DIR}/motus_core_P{pid}.npz"
    export_npz(rots, root_pos, FPS_CIBLE, src_fps, duration, pid, n_persons, out_path)
    NPZ_FILES.append(out_path)
    size_kb = os.path.getsize(out_path) / 1024
    print(f"  → {os.path.basename(out_path)} | {rots.shape[0]} frames @ {FPS_CIBLE} FPS | {size_kb:.1f} KB")

print(f"\n[U-ALPHA] {len(NPZ_FILES)} .npz exportes — Passer a la Cellule 8")
if PENDING_SEGMENTS:
    print(f"[INFO]   {len(PENDING_SEGMENTS)} segment(s) DECA/FrankMocap non traites — a implementer")


In [ ]:
# ══════ CELLULE 8 — TELECHARGEMENT DES .npz ══════

from google.colab import files as colab_files
import numpy as np, os

assert NPZ_FILES, "[ERREUR] Aucun .npz genere — relancer la Cellule 7"

print(f"[U-ALPHA] {len(NPZ_FILES)} fichier(s) .npz prets :")
for npz_path in NPZ_FILES:
    d = np.load(npz_path, allow_pickle=True)
    size_kb = os.path.getsize(npz_path) / 1024
    print(f"  {os.path.basename(npz_path)} | {d['rotations'].shape[0]} frames | {size_kb:.1f} KB")
    print(f"    rotations    : {d['rotations'].shape} {d['rotations'].dtype}")
    print(f"    root_position: {d['root_position'].shape} {d['root_position'].dtype}")
    print(f"    fps          : {int(d['fps'])} | duration : {float(d['duration']):.1f}s")

print("\nTelechargement en cours...")
for npz_path in NPZ_FILES:
    colab_files.download(npz_path)

print("\n[U-ALPHA] Mission accomplie.")
print("Prochain pas : Ouvrir ANIMA_MECHANICUS_GAMMA.ipynb")
print("  → Uploader ces .npz dans la Fregate U-GAMMA (La Forge)")